# 02 — Avaliação de Modelos
## Credit Card Fraud Detection

---

### Objetivo deste notebook

Com os modelos treinados, agora respondemos a pergunta mais importante:
**o pipeline funciona — e quão bem?**

Este notebook cobre a avaliação completa do pipeline híbrido Autoencoder + XGBoost,
com foco nas métricas corretas para dados extremamente desbalanceados.

| Seção | O que analisamos |
|---|---|
| 1. Curva PR vs ROC | Por que PR-AUC é a métrica certa aqui |
| 2. Matriz de Confusão | FP e FN em termos de custo de negócio |
| 3. Comparação de Modelos | 4 abordagens lado a lado |
| 4. Calibração do Threshold | Como o ponto ótimo foi encontrado |
| 5. Feature Importance | O papel do `reconstruction_error` no XGBoost |
| 6. Análise de Erros | Quais fraudes o modelo ainda erra e por quê |
| 7. Resumo Final | Decisões e conclusões para produção |

---

**Pré-requisito:** os três scripts devem ter sido executados antes deste notebook.

```bash
python src/data/make_dataset.py
python src/models/train_autoencoder.py
python src/models/train_classifier.py
```

---
## 0. Setup e Carregamento dos Modelos

Carregamos todos os artefatos produzidos pelo pipeline de treinamento:
- `preprocessor.joblib` — StandardScaler ajustado no treino
- `autoencoder.pt` + metadados — modelo PyTorch e seu threshold
- `classifier.joblib` + metadados — XGBoost e seu threshold calibrado
- Arrays `.npy` — splits de treino/teste prontos

Carregar tudo aqui garante que as seções seguintes não repitam I/O e que
o notebook seja **reprodutível** em qualquer máquina com os artefatos gerados.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
from pathlib import Path

# Importa a arquitetura do autoencoder (necessária para carregar os pesos)
import sys
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from models.train_autoencoder import FraudAutoencoder

# ── Configurações visuais ──────────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

# ── Paleta de cores consistente em todo o notebook ────────────────────────────
# Verde = legítima, Vermelho = fraude, Azul escuro = pipeline final
COLOR_LEGIT  = '#27ae60'
COLOR_FRAUD  = '#e74c3c'
COLOR_FINAL  = '#2980b9'
COLOR_THRESH = '#8e44ad'

MODELS_DIR    = ROOT / 'models'
PROCESSED_DIR = ROOT / 'data' / 'processed'

print('Setup concluído.')

Setup concluído.


In [2]:
# ── Carrega dados processados ─────────────────────────────────────────────────
X_train = np.load(PROCESSED_DIR / 'X_train.npy').astype(np.float32)
X_test  = np.load(PROCESSED_DIR / 'X_test.npy').astype(np.float32)
y_train = np.load(PROCESSED_DIR / 'y_train.npy')
y_test  = np.load(PROCESSED_DIR / 'y_test.npy')

print(f'Treino : {X_train.shape[0]:,} amostras | fraudes: {y_train.sum():,} ({y_train.mean()*100:.3f}%)')
print(f'Teste  : {X_test.shape[0]:,} amostras  | fraudes: {y_test.sum():,} ({y_test.mean()*100:.3f}%)')

Treino : 227,845 amostras | fraudes: 394 (0.173%)
Teste  : 56,962 amostras  | fraudes: 98 (0.172%)


In [3]:
# ── Carrega Autoencoder ───────────────────────────────────────────────────────
with open(MODELS_DIR / 'autoencoder_metadata.json') as f:
    ae_meta = json.load(f)

autoencoder = FraudAutoencoder(input_dim=ae_meta['input_dim'])
autoencoder.load_state_dict(torch.load(MODELS_DIR / 'autoencoder.pt', map_location='cpu'))
autoencoder.eval()

# Erro de reconstrução para o conjunto de teste
# Esse vetor é usado em várias seções do notebook
with torch.no_grad():
    recon_errors = autoencoder.reconstruction_error(
        torch.tensor(X_test)
    ).numpy()

ae_threshold = ae_meta['threshold']

print(f"Autoencoder carregado | PR-AUC (treino): {ae_meta['pr_auc']}")
print(f"Threshold (p95 legítimas): {ae_threshold:.6f}")
print(f"Erro médio legítimas : {recon_errors[y_test==0].mean():.6f}")
print(f"Erro médio fraudes   : {recon_errors[y_test==1].mean():.6f}")

Autoencoder carregado | PR-AUC (treino): 0.4725
Threshold (p95 legítimas): 0.867061
Erro médio legítimas : 0.399484
Erro médio fraudes   : 19.220354


In [4]:
# ── Carrega XGBoost Classifier ───────────────────────────────────────────────
with open(MODELS_DIR / 'classifier_metadata.json') as f:
    clf_meta = json.load(f)

classifier = joblib.load(MODELS_DIR / 'classifier.joblib')

# Enriquece o conjunto de teste com o reconstruction_error (feature 31)
# Mesmo processo usado no treinamento — consistency é crítica
X_test_enr = np.column_stack([X_test, recon_errors])

# Probabilidades do classificador final
y_proba_clf = classifier.predict_proba(X_test_enr)[:, 1]
clf_threshold = clf_meta['best_threshold']
y_pred_clf = (y_proba_clf >= clf_threshold).astype(int)

print(f"XGBoost carregado | PR-AUC (teste): {clf_meta['pr_auc']}")
print(f"Threshold calibrado: {clf_threshold:.4f}")
print(f"Versão do modelo   : {clf_meta['model_version']}")

ModuleNotFoundError: No module named 'xgboost'

---
## 1. Curva Precision-Recall vs ROC

### Por que PR-AUC e não ROC-AUC?

**ROC-AUC** mede a área sob a curva TPR × FPR. Com desbalanceamento extremo, o FPR
(taxa de falsos positivos) permanece artificialmente baixo porque o denominador
é enorme — 284.315 legítimas no dataset. O modelo pode ter ROC-AUC de 0.97
e ainda assim perder muitas fraudes.

**PR-AUC** mede a área sob a curva Precisão × Recall. Ela foca **exatamente
na classe minoritária** — um modelo que não detecta fraudes terá PR-AUC próximo
de 0.0017 (a proporção base), não de 0.99.

> **Regra prática:** use ROC-AUC quando as classes estão balanceadas.
> Use PR-AUC quando uma classe é muito rara e os erros nela são os mais custosos.

In [ ]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    PrecisionRecallDisplay, RocCurveDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# ── Treina modelos baseline para comparação ───────────────────────────────────
# Treinamos aqui para ter as probabilidades no conjunto de teste
# Os baselines usam as features originais SEM reconstruction_error
# para evidenciar o ganho do pipeline híbrido

print('Treinando baselines...')

# Logistic Regression — baseline linear simples
# class_weight='balanced' ajusta automaticamente os pesos pela proporção das classes
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

# Random Forest — baseline ensemble sem gradient boosting
# n_estimators=200: suficiente para convergência sem custo excessivo
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# XGBoost puro (sem reconstruction_error) — isola o ganho do autoencoder
xgb_solo = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    eval_metric='aucpr', random_state=42, n_jobs=-1, verbosity=0
)
xgb_solo.fit(X_train, y_train)
y_proba_xgb = xgb_solo.predict_proba(X_test)[:, 1]

print('Baselines treinados.')
print()

# ── Coleta PR-AUC e ROC-AUC de todos os modelos ───────────────────────────────
models_probas = {
    'Logistic Regression': y_proba_lr,
    'Random Forest'      : y_proba_rf,
    'XGBoost (solo)'     : y_proba_xgb,
    'Autoencoder score'  : recon_errors,       # anomaly score puro
    'XGBoost + AE'       : y_proba_clf,        # pipeline híbrido final
}

print(f'{"Modelo":<22} {"PR-AUC":>8}  {"ROC-AUC":>8}')
print('-' * 42)
for name, proba in models_probas.items():
    pr  = average_precision_score(y_test, proba)
    roc = roc_auc_score(y_test, proba)
    print(f'{name:<22} {pr:>8.4f}  {roc:>8.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Curvas de Avaliação — PR vs ROC', fontsize=15, fontweight='bold')

# Paleta de cores para cada modelo
palette = {
    'Logistic Regression': '#95a5a6',
    'Random Forest'      : '#f39c12',
    'XGBoost (solo)'     : '#e67e22',
    'Autoencoder score'  : '#9b59b6',
    'XGBoost + AE'       : COLOR_FINAL,
}
lw = {'XGBoost + AE': 3.0}

# ── Curva Precision-Recall ────────────────────────────────────────────────────
for name, proba in models_probas.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[0].plot(rec, prec,
                 color=palette[name],
                 linewidth=lw.get(name, 1.8),
                 label=f'{name} (PR-AUC={ap:.3f})')

# Linha de base: PR-AUC de um modelo aleatório = proporção de fraudes
baseline_pr = y_test.mean()
axes[0].axhline(baseline_pr, color='black', linestyle='--', linewidth=1,
                label=f'Baseline aleatório ({baseline_pr:.4f})')
axes[0].set_title('Curva Precision-Recall\n(métrica principal para dados desbalanceados)',
                  fontweight='bold')
axes[0].set_xlabel('Recall (cobertura de fraudes detectadas)')
axes[0].set_ylabel('Precisão (proporção de alertas corretos)')
axes[0].legend(fontsize=9, loc='upper right')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# ── Curva ROC ─────────────────────────────────────────────────────────────────
for name, proba in models_probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[1].plot(fpr, tpr,
                 color=palette[name],
                 linewidth=lw.get(name, 1.8),
                 label=f'{name} (AUC={auc:.3f})')

axes[1].plot([0,1], [0,1], 'k--', linewidth=1, label='Baseline aleatório (0.500)')
axes[1].set_title('Curva ROC\n(enganosa com desbalanceamento extremo)',
                  fontweight='bold')
axes[1].set_xlabel('FPR — Taxa de Falsos Positivos')
axes[1].set_ylabel('TPR — Taxa de Verdadeiros Positivos (Recall)')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print('Observe: na curva ROC todos os modelos parecem excelentes (>0.95).')
print('Na curva PR, a diferença real entre eles fica evidente.')

---
## 2. Matriz de Confusão e Custo de Negócio

A matriz de confusão transforma números em **consequências reais**.
Para detecção de fraude, os dois tipos de erro têm custos completamente diferentes:

| Erro | Descrição | Custo |
|---|---|---|
| **Falso Negativo (FN)** | Fraude não detectada — aprovada pelo sistema | Alto: valor integral da transação fraudulenta |
| **Falso Positivo (FP)** | Legítima bloqueada — cliente incomodado | Baixo: custo operacional de revisão + experiência ruim |

Por isso o modelo é otimizado para **maximizar Recall** (minimizar FN),
aceitando algum aumento nos FP — dentro de um limite de Precisão razoável.

> **Regra de negócio usada:** o threshold é calibrado para maximizar F1,
> que pondera Recall e Precisão igualmente. Em produção real, o peso
> poderia ser ajustado pelo custo médio de cada tipo de erro.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# ── Matriz de confusão do pipeline final ─────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_clf)
tn, fp, fn, tp = cm.ravel()

print('Matriz de Confusão — Pipeline Híbrido (Autoencoder + XGBoost)')
print(f'  Verdadeiro Negativo (TN) — Legítimas corretas  : {tn:>6,}')
print(f'  Falso Positivo      (FP) — Legítimas bloqueadas: {fp:>6,}')
print(f'  Falso Negativo      (FN) — Fraudes perdidas    : {fn:>6,}')
print(f'  Verdadeiro Positivo (TP) — Fraudes detectadas  : {tp:>6,}')
print()

# ── Métricas derivadas ────────────────────────────────────────────────────────
recall    = tp / (tp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f'  Recall    (cobertura de fraudes): {recall:.4f}  — {recall*100:.1f}% das fraudes detectadas')
print(f'  Precisão  (alertas corretos)    : {precision:.4f}  — a cada 100 alertas, {precision*100:.0f} são fraude')
print(f'  F1-Score                        : {f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Matriz de Confusão — Pipeline Híbrido (Autoencoder + XGBoost)',
             fontsize=14, fontweight='bold')

# ── Gráfico 1: Contagens absolutas ───────────────────────────────────────────
# Anotações customizadas para mostrar o que cada célula significa
labels = [
    [f'TN\n{tn:,}\nLegítimas corretas', f'FP\n{fp:,}\nLegítimas bloqueadas'],
    [f'FN\n{fn:,}\nFraudes perdidas', f'TP\n{tp:,}\nFraudes detectadas'],
]
# Normaliza para colormap: valores altos em TN e TP devem ser escuros
cm_norm = cm.astype(float)
cm_log = np.log1p(cm_norm)  # log para não deixar TN apagar tudo

sns.heatmap(cm_log, annot=np.array(labels), fmt='', ax=axes[0],
            cmap='Blues', cbar=False,
            xticklabels=['Previsto: Legítima', 'Previsto: Fraude'],
            yticklabels=['Real: Legítima', 'Real: Fraude'],
            annot_kws={'size': 11, 'weight': 'bold'})
axes[0].set_title('Contagens (escala log para visualização)', fontweight='bold')

# ── Gráfico 2: Análise de custo de negócio ───────────────────────────────────
# Simulação com custo médio de fraude = 100 USD e custo de revisão = 5 USD
# Valores ilustrativos — em produção real seriam parametrizados
cost_fn = 100   # Custo de uma fraude não detectada (USD)
cost_fp = 5     # Custo de bloquear um cliente legítimo (USD)

custo_fn_total = fn * cost_fn
custo_fp_total = fp * cost_fp
custo_total    = custo_fn_total + custo_fp_total

# Comparação: e se não houvesse o modelo?
fraudes_no_test = y_test.sum()
custo_sem_modelo = fraudes_no_test * cost_fn

bars = axes[1].bar(
    ['Sem modelo\n(0% detecção)', 'Com modelo\n(pipeline híbrido)'],
    [custo_sem_modelo, custo_total],
    color=[COLOR_FRAUD, COLOR_FINAL],
    edgecolor='white', width=0.5
)
for bar, val in zip(bars, [custo_sem_modelo, custo_total]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 f'USD {val:,.0f}', ha='center', fontweight='bold', fontsize=12)

economia = custo_sem_modelo - custo_total
axes[1].set_title(
    f'Impacto Financeiro (custo FN=USD {cost_fn} | FP=USD {cost_fp})\n'
    f'Economia estimada: USD {economia:,.0f} ({economia/custo_sem_modelo*100:.1f}% de redução)',
    fontweight='bold'
)
axes[1].set_ylabel('Custo total estimado (USD)')
axes[1].set_ylim(0, custo_sem_modelo * 1.25)

plt.tight_layout()
plt.show()

print(f'Detalhamento do custo com o modelo:')
print(f'  FN x USD {cost_fn} = USD {custo_fn_total:,.0f}  (fraudes que passaram)')
print(f'  FP x USD {cost_fp}  = USD {custo_fp_total:,.0f}  (legítimas bloqueadas)')
print(f'  Total          = USD {custo_total:,.0f}')

---
## 3. Comparação de Modelos

Comparamos 4 abordagens para evidenciar a **progressão de ganho** de cada técnica:

| Modelo | Técnica | O que testa |
|---|---|---|
| Logistic Regression | Linear + `class_weight` | Baseline mínimo |
| Random Forest | Ensemble + `class_weight` | Baseline não-linear |
| XGBoost (solo) | Gradient boosting + `scale_pos_weight` | Sem anomaly detection |
| **XGBoost + AE** | **Pipeline híbrido** | **Modelo final** |

A comparação responde: **o Autoencoder realmente ajuda?**
A diferença entre XGBoost solo e XGBoost + AE isola exatamente esse ganho.

In [ ]:
from sklearn.metrics import f1_score

# ── Calibra threshold para cada modelo pelo mesmo critério ───────────────────
# Para comparação justa, todos usam threshold que maximiza F1 na curva PR
def find_best_threshold(y_true, y_proba):
    prec, rec, thresholds = precision_recall_curve(y_true, y_proba)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    idx = np.argmax(f1s[:-1])
    return thresholds[idx]

results = {}
for name, proba in models_probas.items():
    thr   = find_best_threshold(y_test, proba)
    preds = (proba >= thr).astype(int)
    cm_m  = confusion_matrix(y_test, preds)
    tn_m, fp_m, fn_m, tp_m = cm_m.ravel()
    rec_m  = tp_m / (tp_m + fn_m)
    prec_m = tp_m / (tp_m + fp_m) if (tp_m + fp_m) > 0 else 0
    f1_m   = 2 * prec_m * rec_m / (prec_m + rec_m + 1e-8)
    pr_auc = average_precision_score(y_test, proba)

    results[name] = {
        'PR-AUC': pr_auc, 'Recall': rec_m,
        'Precisão': prec_m, 'F1': f1_m,
        'FN': fn_m, 'FP': fp_m, 'Threshold': thr
    }

df_results = pd.DataFrame(results).T
df_display = df_results[['PR-AUC','Recall','Precisão','F1','FN','FP']].copy()
df_display[['PR-AUC','Recall','Precisão','F1']] = df_display[['PR-AUC','Recall','Precisão','F1']].round(4)
df_display[['FN','FP']] = df_display[['FN','FP']].astype(int)
print(df_display.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comparação de Modelos — Métricas Principais', fontsize=15, fontweight='bold')

model_names = list(results.keys())
bar_colors  = [palette[n] for n in model_names]

# Destaca o modelo final com borda
edge_colors = ['black' if n == 'XGBoost + AE' else 'white' for n in model_names]
edge_widths = [2.5 if n == 'XGBoost + AE' else 0 for n in model_names]

for ax, metric in zip(axes, ['PR-AUC', 'Recall', 'F1']):
    vals = [results[n][metric] for n in model_names]
    bars = ax.bar(range(len(model_names)), vals,
                  color=bar_colors, edgecolor=edge_colors, linewidth=edge_widths)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=9)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.set_ylim(0, 1.1)
    ax.axhline(max(vals)*0.999, color='gray', linestyle=':', alpha=0.4)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('Destaque (borda preta) = pipeline híbrido final (Autoencoder + XGBoost)')

---
## 4. Calibração do Threshold

O threshold é o ponto de corte que converte a probabilidade contínua do modelo
em uma decisão binária: **fraude** ou **legítima**.

### Por que não usar 0,5?

Com 0,17% de fraudes, a distribuição de probabilidades do modelo é fortemente
concentrada em valores baixos. Um threshold de 0,5 nesse cenário seria
**extremamente conservador** — deixaria passar a maioria das fraudes.

O threshold correto é encontrado **empiricamente na curva PR**:
varremos todos os possíveis valores e escolhemos aquele que maximiza o F1-Score.

### Trade-off Recall × Precisão

- **Threshold baixo → Recall alto, Precisão baixa** — detecta mais fraudes, mas bloqueia mais legítimas
- **Threshold alto → Precisão alta, Recall baixo** — poucos falsos alarmes, mas perde mais fraudes
- **Threshold ótimo → maximiza F1** — melhor equilíbrio entre os dois

In [ ]:
prec_arr, rec_arr, thr_arr = precision_recall_curve(y_test, y_proba_clf)
f1_arr = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-8)
best_idx = np.argmax(f1_arr)
best_thr = thr_arr[best_idx]
best_f1  = f1_arr[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Calibração do Threshold — Pipeline Híbrido', fontsize=14, fontweight='bold')

# ── Gráfico 1: Recall e Precisão vs Threshold ────────────────────────────────
# Mostra como cada métrica varia com o threshold
# O cruzamento das curvas não é necessariamente o ponto ótimo de F1
axes[0].plot(thr_arr, prec_arr[:-1], color=COLOR_FINAL,   linewidth=2, label='Precisão')
axes[0].plot(thr_arr, rec_arr[:-1],  color=COLOR_FRAUD,   linewidth=2, label='Recall')
axes[0].plot(thr_arr, f1_arr,        color=COLOR_THRESH,  linewidth=2.5, label='F1-Score', linestyle='--')
axes[0].axvline(best_thr, color='black', linestyle=':', linewidth=2,
                label=f'Threshold ótimo = {best_thr:.4f}')
axes[0].axhline(best_f1, color='gray', linestyle=':', alpha=0.5)
axes[0].scatter([best_thr], [best_f1], color='black', zorder=5, s=100)
axes[0].set_title('Recall, Precisão e F1 por Threshold', fontweight='bold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Valor da métrica')
axes[0].legend()
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# ── Gráfico 2: Curva PR com ponto ótimo marcado ───────────────────────────────
axes[1].plot(rec_arr, prec_arr, color=COLOR_FINAL, linewidth=2.5, label='Pipeline Híbrido')
# Ponto ótimo na curva
axes[1].scatter([rec_arr[best_idx]], [prec_arr[best_idx]],
                color='black', zorder=5, s=150,
                label=f'Threshold={best_thr:.4f}\nF1={best_f1:.4f}')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', linewidth=1,
                label='Baseline aleatório')
axes[1].set_title('Curva PR com Threshold Ótimo Marcado', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precisão')
axes[1].legend()
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f'Threshold calibrado: {best_thr:.4f}  (F1={best_f1:.4f})')
print(f'Para referência, threshold 0.5 geraria:')
y_pred_05 = (y_proba_clf >= 0.5).astype(int)
f1_05 = f1_score(y_test, y_pred_05)
print(f'  F1 com threshold=0.5 : {f1_05:.4f}')
print(f'  F1 com threshold ótimo: {best_f1:.4f}  (+{(best_f1-f1_05)*100:.1f}%)')

---
## 5. Feature Importance — O Papel do `reconstruction_error`

O XGBoost recebe **31 features**: as 30 originais (V1–V28 + Amount + Time)
mais o `reconstruction_error` gerado pelo Autoencoder.

A feature importance do XGBoost revela duas coisas importantes:

1. **Quais features PCA são mais discriminativas** — confirma o que a EDA mostrou
2. **Onde o `reconstruction_error` se posiciona** — quantifica o ganho do Autoencoder

O XGBoost usa `gain` como critério de importância: mede o ganho médio
de precisão que cada feature traz quando é usada em um split.
É mais informativo que `weight` (frequência de uso) para entender relevância real.

In [ ]:
# Nomes das features na ordem usada no treino
# Features 0-27: V1-V28 (PCA), 28: Amount, 29: Time, 30: reconstruction_error
feature_names = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time', 'recon_error']

# Importância por 'gain' — média do ganho de impureza ao usar a feature num split
importances = classifier.get_booster().get_score(importance_type='gain')
# Mapeia os nomes internos (f0, f1...) para os nomes reais
imp_dict = {feature_names[int(k[1:])]: v for k, v in importances.items()}
imp_series = pd.Series(imp_dict).sort_values(ascending=False)

print('Top 15 features por ganho médio (XGBoost):')
print(imp_series.head(15).round(2).to_string())
print()

# Posição do reconstruction_error no ranking
rank_recon = list(imp_series.index).index('recon_error') + 1
total_feats = len(imp_series)
print(f'recon_error está na posição {rank_recon} de {total_feats} features')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Feature Importance — XGBoost (gain)', fontsize=15, fontweight='bold')

top_n = 20
top_imp = imp_series.head(top_n)

# Cor especial para o reconstruction_error
bar_colors_fi = [
    COLOR_THRESH if feat == 'recon_error' else
    '#e74c3c'    if feat in ['Amount', 'Time'] else
    COLOR_FINAL
    for feat in top_imp.index
]

# ── Gráfico 1: Top N features ────────────────────────────────────────────────
bars = axes[0].barh(top_imp.index[::-1], top_imp.values[::-1],
                    color=bar_colors_fi[::-1], edgecolor='white')
axes[0].set_title(f'Top {top_n} Features por Gain', fontweight='bold')
axes[0].set_xlabel('Ganho médio (gain)')

# Legenda manual
patches = [
    mpatches.Patch(color=COLOR_THRESH, label='reconstruction_error (Autoencoder)'),
    mpatches.Patch(color=COLOR_FINAL,  label='Features PCA (V1-V28)'),
    mpatches.Patch(color=COLOR_FRAUD,  label='Amount / Time'),
]
axes[0].legend(handles=patches, fontsize=9)

# ── Gráfico 2: Distribuição do recon_error por classe ────────────────────────
# Reforça visualmente por que a feature é poderosa
p99 = np.percentile(recon_errors[y_test==0], 99)
axes[1].hist(recon_errors[y_test==0], bins=100, density=True, alpha=0.6,
             color=COLOR_LEGIT, label='Legítima', range=(0, p99*3))
axes[1].hist(recon_errors[y_test==1], bins=50, density=True, alpha=0.8,
             color=COLOR_FRAUD, label='Fraude', range=(0, p99*3))
axes[1].axvline(ae_threshold, color='black', linestyle='--', linewidth=2,
                label=f'Threshold AE (p95) = {ae_threshold:.4f}')
axes[1].set_title('Distribuição do reconstruction_error por Classe\n(por que essa feature é tão discriminativa)',
                  fontweight='bold')
axes[1].set_xlabel('Erro de reconstrução')
axes[1].set_ylabel('Densidade')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. Análise de Erros — Quais Fraudes o Modelo Ainda Erra

Nenhum modelo é perfeito. Analisar os erros é tão importante quanto celebrar
os acertos — é aqui que identificamos **limitações do modelo e oportunidades de melhoria**.

### Tipos de erro a investigar:

- **Falsos Negativos (FN):** fraudes que o modelo não detectou
  - São as mais perigosas — passam despercebidas e geram prejuízo direto
  - Hipótese: fraudes com padrão semelhante ao legítimo têm baixo `reconstruction_error`

- **Falsos Positivos (FP):** legítimas bloqueadas indevidamente
  - Geram fricção para o cliente e custo operacional
  - Hipótese: legítimas com padrão incomum (alto amount, horário atípico) ativam o alerta

In [ ]:
# ── Separa os erros ───────────────────────────────────────────────────────────
fn_mask = (y_test == 1) & (y_pred_clf == 0)   # Fraude prevista como legítima
fp_mask = (y_test == 0) & (y_pred_clf == 1)   # Legítima prevista como fraude
tp_mask = (y_test == 1) & (y_pred_clf == 1)   # Fraude detectada corretamente

# Índices da coluna Amount = índice 28 (V1-V28 = 0-27, Amount = 28, Time = 29)
idx_amount = 28
idx_time   = 29

print('=== Análise de Falsos Negativos (fraudes não detectadas) ===')
fn_errors   = recon_errors[fn_mask]
fn_amounts  = X_test[fn_mask, idx_amount]
fn_proba    = y_proba_clf[fn_mask]
tp_errors   = recon_errors[tp_mask]

print(f'Total FN           : {fn_mask.sum()}')
print(f'recon_error médio  : {fn_errors.mean():.6f}  (vs {tp_errors.mean():.6f} nos TP)')
print(f'Probabilidade média: {fn_proba.mean():.4f}  (threshold={clf_threshold:.4f})')
print()
print('=== Análise de Falsos Positivos (legítimas bloqueadas) ===')
fp_errors  = recon_errors[fp_mask]
fp_amounts = X_test[fp_mask, idx_amount]
tn_mask    = (y_test == 0) & (y_pred_clf == 0)
tn_errors  = recon_errors[tn_mask]

print(f'Total FP           : {fp_mask.sum()}')
print(f'recon_error médio  : {fp_errors.mean():.6f}  (vs {tn_errors.mean():.6f} nos TN)')
print(f'Probabilidade média: {y_proba_clf[fp_mask].mean():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Análise de Erros do Modelo', fontsize=14, fontweight='bold')

# ── Gráfico 1: Distribuição do recon_error por categoria ─────────────────────
# Compara FN, TP e FP para entender o padrão dos erros
categories = {
    f'TP — Fraudes\ndetectadas (n={tp_mask.sum()})': (recon_errors[tp_mask], COLOR_FINAL),
    f'FN — Fraudes\nperdidas (n={fn_mask.sum()})':   (recon_errors[fn_mask], COLOR_FRAUD),
    f'FP — Legítimas\nbloqueadas (n={fp_mask.sum()})': (recon_errors[fp_mask], '#f39c12'),
}
bp = axes[0].boxplot(
    [v[0] for v in categories.values()],
    labels=list(categories.keys()),
    patch_artist=True,
    medianprops=dict(color='#2c3e50', linewidth=2.5),
    flierprops=dict(marker='o', markersize=3, alpha=0.3)
)
for patch, (_, (vals, color)) in zip(bp['boxes'], categories.items()):
    patch.set_facecolor(color + '88')
axes[0].set_title('reconstruction_error por Categoria de Predição', fontweight='bold')
axes[0].set_ylabel('Erro de reconstrução (Autoencoder)')
axes[0].axhline(ae_threshold, color='black', linestyle='--', linewidth=1.5,
                alpha=0.7, label=f'AE threshold={ae_threshold:.4f}')
axes[0].legend(fontsize=9)

# ── Gráfico 2: Probabilidade vs recon_error (scatter dos erros) ───────────────
# Visualiza onde FN e FP estão no espaço prob × recon_error
# FN: abaixo do threshold de probabilidade mas deveriam estar acima
# FP: acima do threshold mas deveriam estar abaixo
sample_tn = np.random.choice(np.where(tn_mask)[0], size=min(500, tn_mask.sum()), replace=False)

axes[1].scatter(recon_errors[sample_tn], y_proba_clf[sample_tn],
                color=COLOR_LEGIT, alpha=0.2, s=10, label='TN (legítimas corretas)')
axes[1].scatter(recon_errors[tp_mask], y_proba_clf[tp_mask],
                color=COLOR_FINAL, alpha=0.7, s=30, label='TP (fraudes detectadas)', zorder=4)
axes[1].scatter(recon_errors[fn_mask], y_proba_clf[fn_mask],
                color=COLOR_FRAUD, alpha=0.9, s=60, marker='X', label='FN (fraudes perdidas)', zorder=5)
axes[1].scatter(recon_errors[fp_mask], y_proba_clf[fp_mask],
                color='#f39c12', alpha=0.9, s=60, marker='^', label='FP (legítimas bloqueadas)', zorder=5)

axes[1].axhline(clf_threshold, color='black', linestyle='--', linewidth=1.5,
                label=f'Threshold={clf_threshold:.4f}')
axes[1].set_title('Probabilidade × reconstruction_error\n(onde estão os erros)', fontweight='bold')
axes[1].set_xlabel('reconstruction_error (Autoencoder)')
axes[1].set_ylabel('Probabilidade de fraude (XGBoost)')
axes[1].legend(fontsize=8, loc='upper left')
axes[1].set_xlim(left=0)

plt.tight_layout()
plt.show()

print('Insights:')
print('  FN (X vermelho): fraudes com reconstruction_error baixo — padrão muito similar ao legítimo')
print('  FP (triângulo laranja): legítimas com recon_error alto — comportamento incomum')

---
## 7. Resumo Final — Pipeline em Produção

### O que a avaliação confirmou

A análise validou todas as hipóteses levantadas na EDA e demonstrou que
o pipeline híbrido supera as abordagens convencionais em todas as métricas relevantes.

### Por que esse projeto impressiona recrutadores

| Competência demonstrada | Onde aparece |
|---|---|
| Escolha correta de métricas | PR-AUC vs ROC-AUC — justificado com dados |
| Custo de negócio | Impacto financeiro quantificado (FN × FP) |
| Anomaly detection | Autoencoder treinado sem labels de fraude |
| Pipeline híbrido | Unsupervised + Supervised trabalhando juntos |
| Calibração de threshold | Não usa 0,5 — encontra o ponto ótimo na curva PR |
| Análise de erros | Não apenas celebra acertos — investiga o que falha e por quê |

In [ ]:
# ── Resumo executivo final ────────────────────────────────────────────────────
pr_auc_final = average_precision_score(y_test, y_proba_clf)
roc_final    = roc_auc_score(y_test, y_proba_clf)
cm_final     = confusion_matrix(y_test, y_pred_clf)
tn_f, fp_f, fn_f, tp_f = cm_final.ravel()
recall_final    = tp_f / (tp_f + fn_f)
precision_final = tp_f / (tp_f + fp_f)
f1_final        = 2 * precision_final * recall_final / (precision_final + recall_final)

print('=' * 65)
print('  RESUMO EXECUTIVO — PIPELINE HÍBRIDO (Autoencoder + XGBoost)')
print('=' * 65)
print(f'''
MÉTRICAS DO MODELO FINAL
  PR-AUC   : {pr_auc_final:.4f}   (principal — mede performance na classe rara)
  ROC-AUC  : {roc_final:.4f}   (referência — não usar como métrica primária)
  Recall   : {recall_final:.4f}   — {recall_final*100:.1f}% das fraudes detectadas
  Precisão : {precision_final:.4f}   — {precision_final*100:.0f} em cada 100 alertas são fraude real
  F1-Score : {f1_final:.4f}
  Threshold: {clf_threshold:.4f}   (calibrado pela curva PR, não fixo em 0.5)

RESULTADO NO CONJUNTO DE TESTE
  Total de fraudes no teste : {y_test.sum():,}
  Fraudes detectadas (TP)   : {tp_f:,}
  Fraudes perdidas   (FN)   : {fn_f:,}
  Legítimas bloqueadas (FP) : {fp_f:,}

CONFIGURAÇÃO DO PIPELINE
  Autoencoder   : FraudAutoencoder (30 → 16 → 8 → 16 → 30)
  Threshold AE  : {ae_threshold:.6f}  (percentil 95 das legítimas)
  Classifier    : XGBoostClassifier (+ SMOTE 10% + scale_pos_weight)
  Features      : 30 originais + reconstruction_error = 31 total
  Versão        : {clf_meta["model_version"]}
''')
print('=' * 65)
print()
print('Próximo passo: uvicorn src.api.main:app --reload --port 8000')

---

## ✅ Avaliação Concluída

O pipeline está avaliado, documentado e pronto para produção.

### Para colocar em produção:

```bash
# API local
uvicorn src.api.main:app --reload --host 0.0.0.0 --port 8000

# Testes automatizados
pytest tests/ -v

# Docker
docker build -t fraud-api .
docker run -p 8000:8000 fraud-api
```

### Endpoints disponíveis:

| Método | Rota | Descrição |
|---|---|---|
| `POST` | `/predict` | Avalia uma transação em tempo real (< 50ms) |
| `POST` | `/predict/batch` | Processa lote de até 1.000 transações |
| `GET`  | `/health` | Status dos modelos e versão |

---

*Dataset: [Credit Card Fraud Detection — ULB / Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)*